# 행정안전부_침수흔적도 데이터 전처리

### 데이터 특성
- **침수 단위**: 행정구역이 아닌 '침수구역(폴리곤)' 단위
- **중복 이슈**: 같은 시군구 내 여러 폴리곤이 있어 코드가 반복됨
- **시계열 특성**: 같은 구역이 여러 해에 침수되면 별도 레코드 생성

### 전처리 전략
1. **공간 레벨 중복 판단**: GEOM + 시군구 + 연도/사상 조합으로 유니크 처리
2. **기간 결측값 분리**: 기간 미상 레코드와 정상 레코드 구분
3. **침수 이벤트 집계**: 시군구 + 연도 단위로 통계 생성

In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)  # 모든 컬럼 표시

In [ ]:
RAW_FILE_PATH = '../../../../data/raw/flood_area.csv'
PROCESSED_DIR = '../../../../data/processed'  # 최종 간소화 데이터 저장
STATS_OUTPUT_DIR = './data'  # 통계 파일 저장 (폴리곤, 시군구 집계 등)

START_YEAR = 2016

In [ ]:
# 데이터 로드
flood_data = pd.read_csv(RAW_FILE_PATH)

flood_data.head()

In [ ]:
# 최대 지속기간(370일) 이상치 수동 보정
# - FLDN_END_YMD = 20210806, FLDN_BGNG_YMD = 20200801
# - FLDN_DST_NM = '2020.7.28~8.11침수피해' → 실제 재해 기간에 맞게 수정

mask = (
    (flood_data['FLDN_END_YMD'] == '20210806') &
    (flood_data['FLDN_BGNG_YMD'] == '20200801') &
    (flood_data['FLDN_DST_NM'].fillna('').str.contains('2020.7.28~8.11'))
)

# 재해명에 기재된 기간(2020.7.28~8.11)에 맞춰 시작/종료일 보정
flood_data.loc[mask, 'FLDN_BGNG_YMD'] = '20200728'
flood_data.loc[mask, 'FLDN_END_YMD'] = '20200811'

In [ ]:
flood_data.info()

---
**문제점**: 
- `FLDN_YR`이 '외'로 되어있는 경우 → 외부 작성 자료로 중요 정보 누락
- 숫자가 아닌 값이 포함되어 있어 분석 불가

**처리 로직**:
1. FLDN_YR을 str → int 변환 시도
2. 변환 실패 행은 제거
3. 2016년 이후 데이터만 사용 (최근 데이터 우선)

In [ ]:
# FLDN_YR을 숫자로 변환 (errors='coerce'로 변환 실패 시 NaN 처리)
flood_data['FLDN_YR_num'] = pd.to_numeric(flood_data['FLDN_YR'], errors='coerce')

# 변환 실패 행 제거
flood_data = flood_data.dropna(subset=['FLDN_YR_num'])

In [ ]:
# 2016년 이후 데이터만 필터링
flood_data = flood_data[flood_data['FLDN_YR_num'] >= START_YEAR]

In [ ]:
# 필요는 column drop
columns_to_drop = [
    'FLDN_BGNG_TM',    # 침수 시작 시각
    'FLDN_END_TM',     # 침수 종료 시각  
    'FLDN_CS_DTL_NM',  # 침수 피해 상세
    'FLDN_DST_NM',     # 재해명
    'FLDN_YR',         # 원본 연도 컬럼 (FLDN_YR_num으로 대체)
    'FLDN_GRD'         # 침수 등급
]

flood_data = flood_data.drop(columns=columns_to_drop)

flood_data.head()

In [ ]:
# 결측치 분석

# 결측치 개수 및 비율 계산
missing_stats = pd.DataFrame({
    '결측치_개수': flood_data.isnull().sum(),
    '결측치_비율(%)': (flood_data.isnull().sum() / len(flood_data) * 100).round(2)
})

# 결측치가 있는 컬럼만 출력
missing_stats = missing_stats[missing_stats['결측치_개수'] > 0].sort_values(
    '결측치_개수', ascending=False
)

if len(missing_stats) > 0:
    print(missing_stats)
    print(f"\n결측치가 있는 컬럼: {len(missing_stats)}개")
else:
    print("✓ 결측치 없음")
    
# 전체 데이터 품질 점수
total_values = flood_data.shape[0] * flood_data.shape[1]
missing_values = flood_data.isnull().sum().sum()
completeness = ((total_values - missing_values) / total_values * 100)

print(f"\n[데이터 완전성]")
print(f"  - 전체 셀: {total_values:,}개")
print(f"  - 결측 셀: {missing_values:,}개")
print(f"  - 완전성: {completeness:.2f}%")

### 날짜 데이터 전처리

In [ ]:
# 시작일/종료일을 datetime으로 변환
flood_data['FLDN_BGNG_DATE'] = pd.to_datetime(
    flood_data['FLDN_BGNG_YMD'], 
    format='%Y%m%d', 
    errors='coerce'
)

flood_data['FLDN_END_DATE'] = pd.to_datetime(
    flood_data['FLDN_END_YMD'], 
    format='%Y%m%d', 
    errors='coerce'
)

# 날짜 변환 결과 확인
bgng_missing = flood_data['FLDN_BGNG_DATE'].isna().sum()
end_missing = flood_data['FLDN_END_DATE'].isna().sum()

print(f'시작일 결측: {bgng_missing:,}개 ({bgng_missing/len(flood_data)*100:.2f}%)')
print(f'종료일 결측: {end_missing:,}개 ({end_missing/len(flood_data)*100:.2f}%)')

In [ ]:
# 침수 지속 기간 계산 (일 단위)
# 시작일과 종료일이 모두 있는 경우만 계산
flood_data['duration_days'] = (
    flood_data['FLDN_END_DATE'] - flood_data['FLDN_BGNG_DATE']
).dt.days

# 지속기간이 음수인 경우 수정 (데이터 오류)
negative_duration = (flood_data['duration_days'] < 0).sum()
if negative_duration > 0:
    print(f'지속기간 음수 레코드: {negative_duration}개 (0으로 수정)')
    flood_data.loc[flood_data['duration_days'] < 0, 'duration_days'] = 0

# 지속기간이 없는 경우 1일로 가정 (당일 침수)
flood_data['duration_days'] = flood_data['duration_days'].fillna(1)

print(f"\n✓ 날짜 변환 완료")
print(f"  - 평균 지속기간: {flood_data['duration_days'].mean():.2f}일")
print(f"  - 최대 지속기간: {flood_data['duration_days'].max():.0f}일")
print(f"  - 중앙값: {flood_data['duration_days'].median():.0f}일")

In [ ]:
# 기간 미상 여부를 나타내는 태그 생성
# 시작일 또는 종료일이 없으면 '기간미상', 아니면 '정상'
flood_data['date_status'] = '정상'
flood_data.loc[
    flood_data['FLDN_BGNG_DATE'].isna() | flood_data['FLDN_END_DATE'].isna(),
    'date_status'
] = '기간미상'

# 태그별 레코드 수 확인
date_status_counts = flood_data['date_status'].value_counts()

print("="*60)
print("기간 정보 상태별 분포")
print("="*60)
for status, count in date_status_counts.items():
    pct = count / len(flood_data) * 100
    print(f"  {status}: {count:,}개 ({pct:.2f}%)")

# 기간 미상 레코드의 특성 확인
unknown_dates = flood_data[flood_data['date_status'] == '기간미상']
if len(unknown_dates) > 0:
    print(f"\n[기간 미상 레코드 특성]")
    print(f"  - 평균 침수 면적: {unknown_dates['FLDN_AREA'].mean():,.2f}㎡")
    print(f"  - 연도 분포: {unknown_dates['FLDN_YR_num'].value_counts().to_dict()}")

### 중복 레코드 처리

- [시군구 + 연도 + GEOM(좌표) + 면적]이 모두 동일

In [ ]:
# 중복 판단 키: 시군구 + 연도 + GEOM + 침수면적
dedup_keys = ['STDG_SGG_CD', 'FLDN_YR_num', 'GEOM', 'FLDN_AREA']

# 중복 레코드 확인
duplicates = flood_data.duplicated(subset=dedup_keys, keep=False)
duplicate_count = duplicates.sum()

print(f"중복 레코드: {duplicate_count:,}개")

In [ ]:
# 중복 제거 (첫 번째 레코드만 유지
flood_data_dedup = flood_data.drop_duplicates(subset=dedup_keys, keep='first')
flood_data = flood_data_dedup.copy()

In [ ]:
# 시군구별 집계
regional_stats = flood_data.groupby('STDG_SGG_CD').agg({
    # 침수 발생 빈도
    'SN': 'count',  # 총 침수 레코드 수
    
    # 침수 면적 통계
    'FLDN_AREA': ['sum', 'mean', 'max', 'std'],
    
    # 침수 깊이 통계
    'FLDN_DOWA': ['mean', 'max'],
    
    # 지속 기간 통계
    'duration_days': ['sum', 'mean', 'max'],
    
    # 연도 범위
    'FLDN_YR_num': ['min', 'max', 'nunique']
}).reset_index()

# 컬럼명 단순화
regional_stats.columns = [
    'STDG_SGG_CD',
    'flood_count',  # 침수 발생 횟수
    'total_flood_area', 'avg_flood_area', 'max_flood_area', 'std_flood_area',
    'avg_flood_depth', 'max_flood_depth',
    'total_duration_days', 'avg_duration_days', 'max_duration_days',
    'first_flood_year', 'last_flood_year', 'flood_years_count'
]

# 추가 파생 변수
# 1. 연평균 침수 발생 빈도
regional_stats['annual_flood_frequency'] = (
    regional_stats['flood_count'] / 
    (regional_stats['last_flood_year'] - regional_stats['first_flood_year'] + 1)
)

# 2. 침수 집중도 (특정 연도에 집중되었는지)
regional_stats['flood_concentration'] = (
    regional_stats['flood_count'] / regional_stats['flood_years_count']
)

In [ ]:
print(f"✓ 시군구별 집계 완료")
print(f"  - 집계된 시군구 수: {len(regional_stats)}개")
print(f"  - 생성된 통계 변수: {len(regional_stats.columns)}개")

# 상위 10개 고위험 시군구 (총 침수 면적 기준)
print(f"\n[침수 면적 Top 10 시군구]")
top10_area = regional_stats.nlargest(10, 'total_flood_area')[
    ['STDG_SGG_CD', 'flood_count', 'total_flood_area', 'avg_flood_depth']
]
print(top10_area.to_string(index=False))

# 상위 10개 고빈도 시군구 (침수 발생 횟수 기준)
print(f"\n[침수 발생 빈도 Top 10 시군구]")
top10_freq = regional_stats.nlargest(10, 'flood_count')[
    ['STDG_SGG_CD', 'flood_count', 'avg_flood_area', 'avg_flood_depth']
]
print(top10_freq.to_string(index=False))

In [ ]:
# 저장 경로 설정
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(STATS_OUTPUT_DIR, exist_ok=True)

# 1. 폴리곤 단위 전처리 데이터 (통계용)
polygon_file = os.path.join(STATS_OUTPUT_DIR, 'flood_polygon_processed.csv')
flood_data.to_csv(polygon_file, index=False, encoding='utf-8-sig')

# 2. 시군구 집계 데이터 저장 (통계용)
regional_file = os.path.join(STATS_OUTPUT_DIR, 'flood_regional_stats.csv')
regional_stats.to_csv(regional_file, index=False, encoding='utf-8-sig')

print(f'✅ 통계 파일 저장 완료!')
print(f'  - 폴리곤 데이터: {polygon_file}')
print(f'  - 시군구 집계: {regional_file}')

In [ ]:
# ================================================
# 간소화 데이터셋 생성
# ================================================

# 필요한 컬럼만 추출 및 컬럼명 변경
simple_flood_data = flood_data[['STDG_SGG_CD', 'FLDN_YR_num', 'FLDN_AREA']].copy()

# 침수 발생 월 추출
simple_flood_data['FLOOD_MONTH'] = flood_data['FLDN_BGNG_DATE'].dt.month

# 컬럼명을 sklearn 스타일로 변경
simple_flood_data = simple_flood_data.rename(columns={
    'STDG_SGG_CD': 'SIGUNGU',
    'FLDN_YR_num': 'FLOOD_YEAR',
    'FLDN_AREA': 'FLOOD_AREA'
})

# 컬럼 순서 재정렬
simple_flood_data = simple_flood_data[['SIGUNGU', 'FLOOD_YEAR', 'FLOOD_MONTH', 'FLOOD_AREA']]

# 월 정보가 없는 레코드 제거 (기간 미상 데이터 제외)
before_count = len(simple_flood_data)
simple_flood_data = simple_flood_data.dropna(subset=['FLOOD_MONTH'])
removed_count = before_count - len(simple_flood_data)

# 데이터 타입 최적화
simple_flood_data['SIGUNGU'] = simple_flood_data['SIGUNGU'].astype('int32')
simple_flood_data['FLOOD_YEAR'] = simple_flood_data['FLOOD_YEAR'].astype('int16')
simple_flood_data['FLOOD_MONTH'] = simple_flood_data['FLOOD_MONTH'].astype('int8')
simple_flood_data['FLOOD_AREA'] = simple_flood_data['FLOOD_AREA'].astype('float32')

print("✓ 간소화 데이터셋 생성 완료")
print(f"  - 레코드 수: {len(simple_flood_data):,}개")
print(f"  - 컬럼 수: {len(simple_flood_data.columns)}개")
print(f"  - 메모리: {simple_flood_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n[데이터 미리보기]")
print(simple_flood_data.head(10))

print("\n[기본 통계]")
print(simple_flood_data.describe())

In [ ]:
# ================================================
# save_flood_simple_data() - 간소화 데이터셋 생성 및 저장
# ================================================

def save_flood_simple_data(input_file=None, output_file=None):
    """
    원본 침수 데이터에서 핵심 컬럼만 추출하여 간소화 데이터셋 생성 및 저장
    
    Parameters
    ----------
    input_file : str, optional
        입력 CSV 파일 경로 (기본값: RAW_FILE_PATH)
    
    output_file : str, optional
        출력 CSV 파일 경로 (기본값: PROCESSED_DIR/flood_area.csv)
    
    Returns
    -------
    DataFrame
        저장된 간소화 데이터셋
    
    Examples
    --------
    >>> # 기본 경로로 저장
    >>> data = save_flood_simple_data()
    
    >>> # 커스텀 경로로 저장
    >>> data = save_flood_simple_data(
    ...     input_file='./my_flood_data.csv',
    ...     output_file='./output/simple.csv'
    ... )
    """
    
    # 기본 경로 설정
    if input_file is None:
        input_file = RAW_FILE_PATH
    
    if output_file is None:
        os.makedirs(PROCESSED_DIR, exist_ok=True)
        output_file = os.path.join(PROCESSED_DIR, 'flood_area.csv')
    
    # 원본 데이터 로드
    print(f"📂 원본 데이터 로딩: {input_file}")
    raw_data = pd.read_csv(input_file)
    
    # 연도 필터링 (2016년 이후)
    raw_data['FLDN_YR_num'] = pd.to_numeric(raw_data['FLDN_YR'], errors='coerce')
    filtered_data = raw_data[raw_data['FLDN_YR_num'] >= START_YEAR].copy()
    
    # 날짜 변환
    filtered_data['FLDN_BGNG_DATE'] = pd.to_datetime(
        filtered_data['FLDN_BGNG_YMD'], 
        format='%Y%m%d', 
        errors='coerce'
    )
    
    # 필요한 컬럼만 추출
    simple_data = pd.DataFrame({
        'SIGUNGU': filtered_data['STDG_SGG_CD'].astype('int32'),
        'FLOOD_YEAR': filtered_data['FLDN_YR_num'].astype('int16'),
        'FLOOD_MONTH': filtered_data['FLDN_BGNG_DATE'].dt.month,
        'FLOOD_AREA': filtered_data['FLDN_AREA'].astype('float32')
    })
    
    # 월 정보가 없는 레코드 제거 (기간 미상 데이터 제외)
    before_count = len(simple_data)
    simple_data = simple_data.dropna(subset=['FLOOD_MONTH'])
    removed_count = before_count - len(simple_data)
    if removed_count > 0:
        print(f"⚠️  월 정보 없는 레코드 제거: {removed_count:,}개 ({removed_count/before_count*100:.2f}%)")
    
    # 데이터 타입 최적화
    simple_data['FLOOD_MONTH'] = simple_data['FLOOD_MONTH'].astype('int8')
    
    # 중복 제거 (동일 시군구/연도/면적)
    dedup_keys = ['SIGUNGU', 'FLOOD_YEAR', 'FLOOD_AREA']
    simple_data = simple_data.drop_duplicates(subset=dedup_keys, keep='first')
    
    # CSV 저장
    simple_data.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    print(f"✅ 저장 완료: {output_file}")
    print(f"   - 레코드 수: {len(simple_data):,}개")
    print(f"   - 파일 크기: {os.path.getsize(output_file) / 1024:.2f} KB")
    print(f"   - 시군구 수: {simple_data['SIGUNGU'].nunique()}개")
    print(f"   - 연도 범위: {simple_data['FLOOD_YEAR'].min()}~{simple_data['FLOOD_YEAR'].max()}")
    
    return simple_data


# ================================================
# load_flood_data() - 간소화 데이터셋 로드 (sklearn 스타일)
# ================================================

def load_flood_data(data_file=None, return_X_y=False, as_frame=True):
    """
    침수 데이터를 로드하는 함수 (sklearn.datasets.load_iris 스타일)
    
    Parameters
    ----------
    data_file : str, optional
        데이터 파일 경로 (기본값: PROCESSED_DIR/flood_area.csv)
    
    return_X_y : bool, default=False
        True일 경우 (X, y) 튜플 반환
        False일 경우 전체 DataFrame 반환
    
    as_frame : bool, default=True
        True일 경우 pandas DataFrame 반환
        False일 경우 numpy array 반환
    
    Returns
    -------
    data : DataFrame or ndarray
        return_X_y=False일 때 전체 데이터 반환
    
    (X, y) : tuple of DataFrame or ndarray
        return_X_y=True일 때 반환
        X: SIGUNGU, FLOOD_YEAR, FLOOD_MONTH
        y: FLOOD_AREA
    
    Examples
    --------
    >>> # 전체 데이터 로드
    >>> data = load_flood_data()
    >>> print(data.head())
    
    >>> # 학습용 X, y 분리
    >>> X, y = load_flood_data(return_X_y=True)
    >>> print(X.shape, y.shape)
    
    >>> # NumPy 배열로 로드
    >>> X, y = load_flood_data(return_X_y=True, as_frame=False)
    """
    
    # 기본 경로 설정
    if data_file is None:
        data_file = os.path.join(PROCESSED_DIR, 'flood_area.csv')
    
    # CSV 파일 로드
    if not os.path.exists(data_file):
        raise FileNotFoundError(
            f"❌ 데이터 파일을 찾을 수 없습니다: {data_file}\n"
            f"save_flood_simple_data()를 먼저 실행하세요."
        )
    
    df = pd.read_csv(data_file)
    
    if return_X_y:
        # Feature와 Target 분리
        X = df[['SIGUNGU', 'FLOOD_YEAR', 'FLOOD_MONTH']]
        y = df['FLOOD_AREA']
        
        if not as_frame:
            return X.values, y.values
        return X, y
    
    else:
        # 전체 데이터 반환
        if not as_frame:
            return df.values
        return df


# ================================================
# 함수 실행 예제
# ================================================

print("="*60)
print("침수 데이터 저장/로드 함수")
print("="*60)

print("\n[사용 예제]")
print("""
# 1. 데이터 추출 및 저장
>>> simple_data = save_flood_simple_data()

# 2. 저장된 데이터 로드
>>> data = load_flood_data()
>>> print(data.head())

# 3. X, y 분리하여 로드
>>> X, y = load_flood_data(return_X_y=True)
>>> print(f"X shape: {X.shape}, y shape: {y.shape}")

# 4. NumPy 배열로 로드
>>> X_arr, y_arr = load_flood_data(return_X_y=True, as_frame=False)
>>> print(type(X_arr), type(y_arr))
""")

In [ ]:
# ================================================
# 함수 실행 - 데이터 저장
# ================================================

# 간소화 데이터셋 생성 및 저장
simple_data = save_flood_simple_data()

# 저장된 데이터 미리보기
print("\n[저장된 데이터 미리보기]")
print(simple_data.head(10))

print("\n[기본 통계]")
print(simple_data.describe())

In [ ]:
# ================================================
# 함수 실행 - 데이터 로드 테스트
# ================================================

print("="*60)
print("데이터 로드 함수 테스트")
print("="*60)

# 1. 전체 데이터 로드
print("\n[1] 전체 데이터 로드")
data = load_flood_data()
print(f"✓ Shape: {data.shape}")
print(data.head())

# 2. X, y 분리 (DataFrame)
print("\n[2] X, y 분리 (DataFrame)")
X, y = load_flood_data(return_X_y=True)
print(f"✓ X shape: {X.shape}, y shape: {y.shape}")
print(f"✓ X columns: {list(X.columns)}")

# 3. NumPy 배열로 로드
print("\n[3] NumPy 배열로 로드")
X_arr, y_arr = load_flood_data(return_X_y=True, as_frame=False)
print(f"✓ X type: {type(X_arr)}, shape: {X_arr.shape}")
print(f"✓ y type: {type(y_arr)}, shape: {y_arr.shape}")

print("\n" + "="*60)
print("✅ 모든 테스트 완료!")
print("="*60)